# 01d Actual Dataset, Station Split

This notebook runs Experiment 4 for the journal article: leave-one-station-out validation on the actual manually labelled dataset.

The intent is to test whether PyNRPF can generalize to an unseen real station. For each fold, M8 trains on all other actual stations and is evaluated on the held-out actual station across all dates.

This experiment is likely one of the most important journal additions because it tests spatial/station transfer rather than future-time transfer only.

## Databricks Dependency Setup

Run this cell before the import/setup cells when executing the notebook in Databricks. The helper module imports `yaml`, and full M8 training also needs `xgboost`, `holidays`, and `scikit-learn`. Keeping the install cell inside every experiment notebook makes each notebook runnable on a fresh Databricks cluster without depending on cluster-level library setup.

After `%pip install` finishes, Databricks may ask to restart the Python kernel. If that happens, restart and then continue from the next cell.


In [ ]:
%pip install PyYAML "xgboost>=2,<3" "holidays>=0.40,<1" "scikit-learn>=1.4,<2"


## Imports And Path Setup

This section locates the journal article folder and imports the shared experiment helpers.

The helper path search avoids hard-coding a machine-specific repository path. It should work as long as the notebook is run somewhere inside the PyNRPF repository.

If this cell fails, inspect the current working directory and the presence of `_experiment_helpers.py`.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
helper_dir = None
for candidate in [start, *start.parents]:
    direct = candidate if candidate.name == "notebooks" else candidate / "publication" / "2_journal_article" / "notebooks"
    if (direct / "_experiment_helpers.py").exists():
        helper_dir = direct
        break
if helper_dir is None:
    raise RuntimeError("Could not locate _experiment_helpers.py")
sys.path.insert(0, str(helper_dir))

from _experiment_helpers import (
    dataset_summary,
    experiment_output_dir,
    find_article_root,
    load_config,
    load_dataset,
    run_station_split_experiment,
    station_folds,
)

ARTICLE_ROOT = find_article_root(start)
NOTEBOOK_NAME = "01d_actual_station_split.ipynb"
EXPERIMENT_ID = "experiment_4_actual_station_split"
DATASET_KEY = "actual"
ARTICLE_ROOT

## Load YAML Config

Editable constants live in `config/experiment_config.yaml` rather than being hard-coded in this notebook.

Important values loaded from YAML include dataset paths, split dates, active methods, M8 hyperparameters and thresholds, M7 threshold settings, resume behavior, evaluation rules, and output folder names.

`methods.enabled` is the only place to choose methods. Use `["m7_dtr"]` to run M7 only, or `["m8_xgb", "m7_dtr"]` to run both. When `m8_xgb` is absent, the helper does not train M8 at all.

For the journal article, interval detection is gated by day detection. Interval flags are only allowed on method-predicted positive days. Interval metrics are then evaluated only on TP days, meaning site-days where both `pred_day` and `label_day` are true.

The most important safety flag is `execution.run_full_experiment`. When it is `false`, the notebook performs smoke validation only. When it is `true`, it can train models and write full CSV outputs. `execution.overwrite_outputs: true` is currently intentional so existing checkpoints are regenerated under the corrected interval metric definition.


In [ ]:
cfg = load_config(ARTICLE_ROOT)
active_methods = cfg["methods"]["enabled"]
print("Config:", cfg["_config_path"])
print("Run full experiment:", cfg["execution"]["run_full_experiment"])
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("Interval prediction day scope:", cfg["evaluation"].get("interval_prediction_day_scope"))
print("Interval metric scope:", cfg["evaluation"].get("interval_metric_scope"))
print("Interval metric level:", cfg["evaluation"].get("interval_metric_level_name"))
print("Output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))


## Load And Validate Dataset

This reads the processed actual CSV and validates that the station/timestamp keys and labels are usable.

Validation includes timestamp parsing, duplicate key checks, boolean label coercion, and `label_day` recomputation.

Because this is a station split, pay close attention to the number of stations and positive labels. If one station has very few positives, its fold may have unstable precision/recall.

In [ ]:
df = load_dataset(ARTICLE_ROOT, cfg, DATASET_KEY)
dataset_summary(df, cfg, DATASET_KEY)

## Define Station Folds

All actual stations are retained as held-out folds.

The expected fold count is 8: `act_A` through `act_H`. Each fold trains M8 on the other 7 stations and evaluates on the held-out station across all dates.

If the fold count differs, inspect the anonymized actual dataset and station IDs.

In [ ]:
folds = station_folds(df, cfg, DATASET_KEY)
print("Retained folds:", folds)
print("Number of retained folds:", len(folds))

## Method Helpers

M8 and M7 execution is implemented in the shared helper module to keep this notebook readable while keeping all experiment notebooks consistent.

Active methods come from `methods.enabled` in YAML. The notebook does not maintain a second method list. If the list is only `["m7_dtr"]`, M7 runs independently and no M8 feature building or XGBoost training is attempted. If `m8_xgb` is enabled, M8 training happens only for fold/method tasks that are missing a complete checkpoint unless overwrite mode clears those checkpoints for regeneration.

M8 is the trainable two-stage XGBoost method: first a day-level classifier, then an interval-level classifier inside predicted-positive days. The journal helper ignores any broader M8 review setting and applies a final safety gate so `pred_interval` cannot be true where `pred_day` is false.

M7 is the deterministic threshold-rule baseline and does not train. The underlying legacy helper may produce relaxed interval flags on days that are not strict day positives, so the journal helper masks M7 interval outputs to predicted-positive M7 days before writing predictions or metrics.

Both methods use fixed conference-paper settings from YAML. This is deliberate: the journal experiments compare generalization settings, not retuned parameter sets.


In [ ]:
active_methods = cfg["methods"]["enabled"]
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("M7 enabled:", "m7_dtr" in active_methods)
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("Interval detection gate:", cfg["evaluation"].get("interval_prediction_day_scope"))
print("Interval metric scope:", cfg["evaluation"].get("interval_metric_scope"))
print("M8 thresholds:", cfg["m8_xgb"]["xgb1_day"]["threshold"], cfg["m8_xgb"]["xgb2_timestamp"]["threshold"])


## Run Experiment

Smoke mode writes a manifest only and skips model training.

Full mode trains one M8 model per held-out actual station, so this may be runtime-heavy. Run it only after confirming the folds and config are correct.

Full outputs include per-fold prediction CSVs, per-fold metrics, and pooled station-CV metrics.

In full mode, each fold/method task writes three checkpoint files as soon as that task completes: a prediction CSV under `predictions/`, a per-task metrics CSV under `metrics/`, and a completion YAML under `status/`. The completion marker is written last, so interrupted or failed tasks rerun cleanly.

Because `execution.overwrite_outputs` is currently true, expected fold/method checkpoint files are cleared at the start of a full run and regenerated with the corrected interval policy. The notebook still rebuilds `metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` from completed task files each time, so partial output remains inspectable if a long run stops halfway.

The interval metric row is named `interval_tp_days_only`. It uses daytime rows only and then keeps only TP days where both `pred_day` and `label_day` are true.


In [ ]:
run_station_split_experiment(ARTICLE_ROOT, cfg, DATASET_KEY, EXPERIMENT_ID, NOTEBOOK_NAME)

## Quick Review

Use this final cell to confirm where the notebook wrote outputs.

After a smoke run, expect `manifest.yaml` plus created `predictions/`, `metrics/`, and `status/` folders. After a full or partial run, inspect `status/` first to see which fold/method tasks completed, then inspect the per-task metrics and prediction CSVs.

`metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` are rebuilt from completed task files on every run. If results look wrong later, start debugging by checking the status YAML for the affected fold/method, then compare the manifest settings against `experiment_config.yaml`.


In [ ]:
print("Review output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))